# Concept Bottleneck Models for Interpretable ECG Diagnosis — 02: Black-box baseline

Reference model for the CBM comparison: a 1D ResNet trained end-to-end on 12-lead ECGs
(PTB-XL v1.0.3, 100 Hz, 10 s) to predict the 5 diagnostic superclasses (multi-label).

- **Encoder** shared with the CBM (notebook 03) so that the only difference is the concept bottleneck.
- **Preprocessing:** per-lead standardization with statistics from the training folds only.
- **Protocol:** official folds (1–8 train, 9 val, 10 test), early stopping on validation macro AUROC,
  5 random seeds, mean ± std on test + bootstrap 95% CI.
- **Reference:** Strodthoff et al. (2021) report ≈0.93 macro AUROC on this task.

## 1. Setup
Imports, locates the cached arrays from notebook 01, selects GPU and defines the seeding helper for reproducibility.

In [1]:
import glob, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

hits = glob.glob("/kaggle/input/**/X100.npy", recursive=True)
assert hits, "Add notebook 01 output via '+ Add Input'"
IN = Path(hits[0]).parent
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("IN:", IN, "| device:", DEVICE, torch.cuda.get_device_name(0) if AMP else "")

SUPER = ["NORM", "MI", "STTC", "CD", "HYP"]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

IN: /kaggle/input/notebooks/antonioboza/ptbxl-cbm-01-data-and-concepts | device: cuda Tesla T4


## 2. Data, splits and standardization
Official PTB-XL folds: 1–8 train, 9 validation, 10 test. Per-lead mean and std are computed **on the training set only** and applied to all splits, to avoid information leakage. Signals are transposed to channels-first (N, 12, 1000) for `Conv1d`.

In [2]:
X = np.load(IN / "X100.npy", mmap_mode="r")
L = np.load(IN / "labels.npz")
Y, has_y, fold = L["Y"], L["has_y"], L["fold"]

idx = {
    "train": np.where(has_y & (fold <= 8))[0],
    "val":   np.where(has_y & (fold == 9))[0],
    "test":  np.where(has_y & (fold == 10))[0],
}
print({k: len(v) for k, v in idx.items()})

# Per-lead statistics from TRAIN only -> no information from val/test leaks into preprocessing
Xtr = np.asarray(X[idx["train"]])
mu = Xtr.mean(axis=(0, 1), keepdims=True)
sd = Xtr.std(axis=(0, 1), keepdims=True)
del Xtr

def prep(ii):  # (N, 1000, 12) -> standardized (N, 12, 1000), channels-first for Conv1d
    return ((np.asarray(X[ii]) - mu) / sd).astype(np.float32).transpose(0, 2, 1).copy()

data = {k: (torch.from_numpy(prep(v)), torch.from_numpy(Y[v].astype(np.float32))) for k, v in idx.items()}
assert all(torch.isfinite(d[0]).all() for d in data.values())
np.savez(OUT / "norm_stats.npz", mu=mu, sd=sd)
print({k: tuple(v[0].shape) for k, v in data.items()})

{'train': 17084, 'val': 2146, 'test': 2158}
{'train': (17084, 12, 1000), 'val': (2146, 12, 1000), 'test': (2158, 12, 1000)}


## 3. Model
A 1D ResNet **encoder** (4 stages, kernel size 9, global average + max pooling) followed by a linear head. The encoder is reused unchanged in the CBM, so the bottleneck is the only difference between both models.

In [3]:
class ResBlock1d(nn.Module):
    def __init__(self, cin, cout, stride, k=9, p_drop=0.1):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False);     self.b2 = nn.BatchNorm1d(cout)
        self.drop = nn.Dropout(p_drop)
        self.skip = (nn.Identity() if cin == cout and stride == 1 else
                     nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout)))

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(self.drop(h)))
        return F.relu(h + self.skip(x))


class Encoder(nn.Module):
    """ECG (B, 12, 1000) -> feature vector. Identical in black-box and CBM for a fair comparison."""
    def __init__(self, cin=12, widths=(64, 128, 192, 256), k=9):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(cin, widths[0], 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(widths[0]), nn.ReLU())
        blocks, c = [], widths[0]
        for w in widths:
            blocks += [ResBlock1d(c, w, stride=2, k=k), ResBlock1d(w, w, stride=1, k=k)]
            c = w
        self.blocks = nn.Sequential(*blocks)
        self.out_dim = 2 * c  # concat of global average and global max pooling

    def forward(self, x):
        h = self.blocks(self.stem(x))
        return torch.cat([h.mean(-1), h.amax(-1)], dim=1)


class BlackBox(nn.Module):
    def __init__(self, n_out=5, p_drop=0.3):
        super().__init__()
        self.enc = Encoder()
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(self.enc.out_dim, n_out))

    def forward(self, x):
        return self.head(self.enc(x))


print(f"params: {sum(p.numel() for p in BlackBox().parameters()) / 1e6:.2f} M")

params: 4.20 M


## 4. Training and evaluation functions
AdamW + OneCycle learning-rate schedule, mixed precision, binary cross-entropy (multi-label). Early stopping keeps the checkpoint with the best **validation macro AUROC**; the test set is never used for model selection.

In [4]:
WIN, STRIDE = 250, 125  # 2.5 s windows / 1.25 s stride at 100 Hz, as in the PTB-XL benchmark

def random_crop(xb, win=WIN):
    """Independent random 2.5 s window per sample (data augmentation)."""
    starts = torch.randint(0, xb.shape[-1] - win + 1, (xb.shape[0],), device=xb.device)
    idx = (starts[:, None] + torch.arange(win, device=xb.device))[:, None, :].expand(-1, xb.shape[1], -1)
    return torch.gather(xb, 2, idx)

@torch.no_grad()
def predict(model, Xt, bs=512, crop=False):
    """crop=False: full 10 s input. crop=True: average of sigmoid outputs over sliding 2.5 s windows."""
    model.eval(); out = []
    starts = list(range(0, Xt.shape[-1] - WIN + 1, STRIDE)) if crop else [None]
    for i in range(0, len(Xt), bs):
        xb = Xt[i:i + bs].to(DEVICE)
        with torch.autocast("cuda", enabled=AMP):
            ps = [torch.sigmoid(model(xb if s is None else xb[:, :, s:s + WIN])).float() for s in starts]
        out.append(torch.stack(ps).mean(0).cpu())
    return torch.cat(out).numpy()

def macro_auc(y, p):
    return roc_auc_score(y, p, average="macro")

def train_blackbox(seed, epochs=40, bs=128, lr=1e-3, wd=1e-2, patience=8, crop=False, verbose=True):
    set_seed(seed)
    model = BlackBox().to(DEVICE)
    (Xtr, Ytr), (Xva, Yva) = data["train"], data["val"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((len(Xtr) + bs - 1) // bs))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    g = torch.Generator().manual_seed(seed)
    best, best_state, bad, hist = -1.0, None, 0, []

    for ep in range(epochs):
        model.train(); t0, tot = time.time(), 0.0
        perm = torch.randperm(len(Xtr), generator=g)
        for i in range(0, len(Xtr), bs):
            b = perm[i:i + bs]
            xb, yb = Xtr[b].to(DEVICE, non_blocking=True), Ytr[b].to(DEVICE)
            if crop:
                xb = random_crop(xb)
            with torch.autocast("cuda", enabled=AMP):
                loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(b)

        va = macro_auc(Yva.numpy(), predict(model, Xva, crop=crop))
        hist.append({"epoch": ep, "train_loss": tot / len(Xtr), "val_auc": va})
        if va > best:
            best, bad = va, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose:
            print(f"seed {seed} | ep {ep:02d} | loss {tot / len(Xtr):.4f} | val AUROC {va:.4f} | {time.time() - t0:.1f}s")
        if bad >= patience:
            break

    model.load_state_dict(best_state)
    return model, best, pd.DataFrame(hist)

## 5. Smoke test
Two epochs on one seed to verify the pipeline runs end to end before launching the full experiment.

In [5]:
_m, _best, _h = train_blackbox(seed=0, epochs=2)
print("smoke test OK, val AUROC:", round(_best, 4))
del _m

seed 0 | ep 00 | loss 0.4451 | val AUROC 0.8280 | 5.2s
seed 0 | ep 01 | loss 0.3088 | val AUROC 0.9069 | 3.6s
smoke test OK, val AUROC: 0.9069


## 6. Full experiment (5 seeds)
Trains the model with 5 random seeds and evaluates each on the test fold. Saves the weights and the test probabilities for the paired comparison with the CBM.

In [6]:
SEEDS = [0, 1, 2, 3, 4]
Xte, Yte = data["test"]; yte = Yte.numpy()
rows, test_probs, hists = [], [], []

for s in SEEDS:
    model, best_val, hist = train_blackbox(s)
    p = predict(model, Xte)
    per = roc_auc_score(yte, p, average=None)
    rows.append({"seed": s, "val_auc": best_val, "test_macro_auc": per.mean(),
                 **{f"auc_{c}": a for c, a in zip(SUPER, per)}})
    test_probs.append(p); hists.append(hist.assign(seed=s))
    torch.save(model.state_dict(), OUT / f"blackbox_seed{s}.pt")

res = pd.DataFrame(rows)
res.round(4)

seed 0 | ep 00 | loss 0.5526 | val AUROC 0.8121 | 3.6s
seed 0 | ep 01 | loss 0.3998 | val AUROC 0.8721 | 3.6s
seed 0 | ep 02 | loss 0.3395 | val AUROC 0.8930 | 3.6s
seed 0 | ep 03 | loss 0.3159 | val AUROC 0.8952 | 3.6s
seed 0 | ep 04 | loss 0.2997 | val AUROC 0.8935 | 3.6s
seed 0 | ep 05 | loss 0.2872 | val AUROC 0.9086 | 3.6s
seed 0 | ep 06 | loss 0.2793 | val AUROC 0.8985 | 3.7s
seed 0 | ep 07 | loss 0.2736 | val AUROC 0.9075 | 3.7s
seed 0 | ep 08 | loss 0.2652 | val AUROC 0.9097 | 3.7s
seed 0 | ep 09 | loss 0.2594 | val AUROC 0.9129 | 3.7s
seed 0 | ep 10 | loss 0.2502 | val AUROC 0.9074 | 3.7s
seed 0 | ep 11 | loss 0.2440 | val AUROC 0.9016 | 3.7s
seed 0 | ep 12 | loss 0.2377 | val AUROC 0.9111 | 3.7s
seed 0 | ep 13 | loss 0.2293 | val AUROC 0.9150 | 3.7s
seed 0 | ep 14 | loss 0.2215 | val AUROC 0.9093 | 3.7s
seed 0 | ep 15 | loss 0.2117 | val AUROC 0.9021 | 3.8s
seed 0 | ep 16 | loss 0.1983 | val AUROC 0.9067 | 3.8s
seed 0 | ep 17 | loss 0.1881 | val AUROC 0.9050 | 3.8s
seed 0 | e

,seed,val_auc,test_macro_auc,auc_NORM,auc_MI,auc_STTC,auc_CD,auc_HYP
0,0,0.9150,0.9072,0.9340,0.8884,0.9296,0.8970,0.8869
1,1,0.9176,0.9147,0.9398,0.8968,0.9286,0.9171,0.8915
2,2,0.9139,0.9067,0.9375,0.8900,0.9218,0.9057,0.8788
3,3,0.9162,0.9110,0.9343,0.9093,0.9177,0.9078,0.8858
4,4,0.9131,0.9112,0.9339,0.9081,0.9161,0.9081,0.8900


## 7. Results
Mean ± std across seeds, per-class AUROC and a bootstrap 95% confidence interval for the macro AUROC.

In [7]:
summary = res.drop(columns="seed").agg(["mean", "std"]).T.round(4)
print(summary)

# Bootstrap 95% CI of the macro AUROC on the test set (seed-averaged probabilities)
rng = np.random.default_rng(0)
p_mean = np.mean(test_probs, axis=0)
boots = []
for _ in range(1000):
    ii = rng.integers(0, len(yte), len(yte))
    if yte[ii].sum(0).min() > 0:
        boots.append(macro_auc(yte[ii], p_mean[ii]))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\nSeed-averaged test macro AUROC: {macro_auc(yte, p_mean):.4f}  (95% CI {lo:.4f}–{hi:.4f})")

res.to_csv(OUT / "blackbox_results.csv", index=False)
pd.concat(hists).to_csv(OUT / "blackbox_history.csv", index=False)
np.save(OUT / "blackbox_test_probs.npy", np.array(test_probs))  # needed for paired comparison with CBM

                  mean     std
val_auc         0.9152  0.0018
test_macro_auc  0.9102  0.0033
auc_NORM        0.9359  0.0026
auc_MI          0.8985  0.0098
auc_STTC        0.9227  0.0062
auc_CD          0.9071  0.0072
auc_HYP         0.8866  0.0049

Seed-averaged test macro AUROC: 0.9233  (95% CI 0.9159–0.9305)


## 8a. Exploratory test: random 2.5 s crops (1 seed, validation only)
The PTB-XL benchmark trains on random 2.5 s windows and averages predictions over sliding windows at test time. Decision rule fixed **before** running: adopt cropping if validation macro AUROC > 0.925 (baseline: 0.9150). The test set is not used for this decision.

In [8]:
# Decision rule fixed in advance: adopt random cropping if val macro AUROC > 0.925 (baseline seed 0: 0.9150).
# Test set is NOT evaluated here, so the choice cannot be biased by test results.
m_crop, best_crop, h_crop = train_blackbox(seed=0, epochs=60, patience=10, crop=True)
print(f"\nVal AUROC with random crops: {best_crop:.4f}  vs  full-length baseline: 0.9150")

seed 0 | ep 00 | loss 0.5507 | val AUROC 0.8042 | 2.7s
seed 0 | ep 01 | loss 0.4280 | val AUROC 0.8614 | 2.4s
seed 0 | ep 02 | loss 0.3755 | val AUROC 0.8871 | 2.4s
seed 0 | ep 03 | loss 0.3513 | val AUROC 0.8908 | 2.5s
seed 0 | ep 04 | loss 0.3335 | val AUROC 0.8923 | 2.5s
seed 0 | ep 05 | loss 0.3177 | val AUROC 0.9064 | 2.4s
seed 0 | ep 06 | loss 0.3102 | val AUROC 0.9022 | 2.4s
seed 0 | ep 07 | loss 0.3021 | val AUROC 0.9082 | 2.5s
seed 0 | ep 08 | loss 0.2942 | val AUROC 0.9162 | 2.5s
seed 0 | ep 09 | loss 0.2894 | val AUROC 0.9131 | 2.5s
seed 0 | ep 10 | loss 0.2827 | val AUROC 0.9099 | 2.5s
seed 0 | ep 11 | loss 0.2782 | val AUROC 0.9149 | 2.5s
seed 0 | ep 12 | loss 0.2783 | val AUROC 0.9149 | 2.4s
seed 0 | ep 13 | loss 0.2738 | val AUROC 0.9180 | 2.5s
seed 0 | ep 14 | loss 0.2689 | val AUROC 0.9216 | 2.5s
seed 0 | ep 15 | loss 0.2677 | val AUROC 0.9163 | 2.4s
seed 0 | ep 16 | loss 0.2640 | val AUROC 0.9175 | 2.4s
seed 0 | ep 17 | loss 0.2609 | val AUROC 0.9212 | 2.4s
seed 0 | e

## 8b. Reusable experiment runner
Trains one model per seed, evaluates on the test fold and saves weights, test probabilities and training histories under a tag. Includes a bootstrap helper for confidence intervals.

In [9]:
def bootstrap_ci(y, p, n=1000, seed=0):
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n):
        ii = rng.integers(0, len(y), len(y))
        if y[ii].sum(0).min() > 0:
            boots.append(macro_auc(y[ii], p[ii]))
    return np.percentile(boots, [2.5, 97.5])

def run_experiment(tag, crop, seeds=(0, 1, 2, 3, 4), **kw):
    """Train one model per seed, evaluate on test, save weights / probabilities / histories under `tag`."""
    Xte, Yte = data["test"]; yte = Yte.numpy()
    rows, probs, hists = [], [], []
    for s in seeds:
        model, best_val, hist = train_blackbox(s, crop=crop, verbose=False, **kw)
        p = predict(model, Xte, crop=crop)
        per = roc_auc_score(yte, p, average=None)
        rows.append({"seed": s, "val_auc": best_val, "test_macro_auc": per.mean(), "epochs": len(hist),
                     **{f"auc_{c}": a for c, a in zip(SUPER, per)}})
        probs.append(p); hists.append(hist.assign(seed=s))
        torch.save(model.state_dict(), OUT / f"{tag}_seed{s}.pt")
        print(f"[{tag}] seed {s} | val {best_val:.4f} | test {per.mean():.4f} | {len(hist)} epochs")
    res, probs = pd.DataFrame(rows), np.array(probs)
    res.to_csv(OUT / f"{tag}_results.csv", index=False)
    pd.concat(hists).to_csv(OUT / f"{tag}_history.csv", index=False)
    np.save(OUT / f"{tag}_test_probs.npy", probs)
    return res, probs

del m_crop; torch.cuda.empty_cache()

## 9. Full experiment with random crops (5 seeds)
Validation AUROC with crops (0.928) exceeded the pre-set threshold, so cropping is adopted. This configuration is the final black-box baseline for the CBM comparison.

In [10]:
res_crop, probs_crop = run_experiment("blackbox_crop", crop=True, epochs=60, patience=10)

[blackbox_crop] seed 0 | val 0.9282 | test 0.9210 | 42 epochs
[blackbox_crop] seed 1 | val 0.9281 | test 0.9272 | 46 epochs
[blackbox_crop] seed 2 | val 0.9283 | test 0.9254 | 51 epochs
[blackbox_crop] seed 3 | val 0.9264 | test 0.9242 | 32 epochs
[blackbox_crop] seed 4 | val 0.9269 | test 0.9237 | 46 epochs


## 10. Ablation: full-length input vs. random crops
Single-model mean ± std, 5-seed ensemble with bootstrap 95% CI, per-class AUROC, and a **paired bootstrap** of the ensemble difference (same resampled test records for both models). A CI excluding 0 indicates a significant difference.

In [11]:
yte = data["test"][1].numpy()
res_full = pd.read_csv(OUT / "blackbox_results.csv")
probs_full = np.load(OUT / "blackbox_test_probs.npy")

rows = []
for name, res, probs in [("full 10 s", res_full, probs_full), ("random 2.5 s crops", res_crop, probs_crop)]:
    ens = probs.mean(0); lo, hi = bootstrap_ci(yte, ens)
    rows.append({"setup": name,
                 "single model (mean ± std)": f"{res.test_macro_auc.mean():.4f} ± {res.test_macro_auc.std():.4f}",
                 "5-seed ensemble": f"{macro_auc(yte, ens):.4f} [{lo:.4f}–{hi:.4f}]",
                 **{c: round(res[f'auc_{c}'].mean(), 4) for c in SUPER}})
print(pd.DataFrame(rows).to_string(index=False))

# Paired bootstrap on the ensemble difference (same resampled test records for both models)
rng = np.random.default_rng(1); diffs = []
ef, ec = probs_full.mean(0), probs_crop.mean(0)
for _ in range(1000):
    ii = rng.integers(0, len(yte), len(yte))
    if yte[ii].sum(0).min() > 0:
        diffs.append(macro_auc(yte[ii], ec[ii]) - macro_auc(yte[ii], ef[ii]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nΔ ensemble AUROC (crops − full): {np.mean(diffs):+.4f}  (95% CI {lo:+.4f} to {hi:+.4f})")

             setup single model (mean ± std)        5-seed ensemble   NORM     MI   STTC     CD    HYP
         full 10 s           0.9102 ± 0.0033 0.9233 [0.9159–0.9305] 0.9359 0.8985 0.9227 0.9071 0.8866
random 2.5 s crops           0.9243 ± 0.0023 0.9325 [0.9260–0.9392] 0.9469 0.9205 0.9358 0.9173 0.9011

Δ ensemble AUROC (crops − full): +0.0092  (95% CI +0.0063 to +0.0119)
